# Breast Cancer — Data Generation (Multivariate)

this notebook loads the Breast Cancer Wisconsin dataset from sklearn, and generates three datasets with MCAR, MAR, and MNAR missingness using mdatagen.

run this notebook once to reproduce the CSV files used by `../analysis_multi.ipynb`. the CSVs are saved in the same `data/` folder as this notebook.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer

from mdatagen.multivariate.mMCAR import mMCAR
from mdatagen.multivariate.mMAR  import mMAR
from mdatagen.multivariate.mMNAR import mMNAR

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# initial data set loading
data = load_breast_cancer(as_frame=True)
X_raw = data.data
y = data.target.to_numpy()

X_raw.isna().sum()

X_raw.shape

mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
dtype: int64

(569, 30)

The dataset is already complete with no missing values and all columns are numeric, so we can use it directly.

Now we have a clean numeric dataset to work with. We inject missing values into multiple columns simultaneously to simulate realistic multivariate missingness patterns.

We use `target` (diagnosis: 0=malignant, 1=benign) as the target variable `y`. It is passed to the generators because the library requires it — `target` is not a feature in `X` and will not receive any missing values.

In [3]:
X = X_raw.copy()

### MCAR generation

Here we'll generate a dataset with MCAR missingness across multiple columns simultaneously.

Missingness is completely random and does not depend on any observed variable or on `target`.

In [4]:
np.random.seed(42)
df_mcar = mMCAR(X=X, y=y, missing_rate=30).random()

df_mcar.isna().sum()

df_mcar.shape

mean radius                167
mean texture               177
mean perimeter             167
mean area                  180
mean smoothness            182
mean compactness           179
mean concavity             158
mean concave points        160
mean symmetry              177
mean fractal dimension     180
radius error               157
texture error              167
perimeter error            163
area error                 179
smoothness error           170
compactness error          156
concavity error            174
concave points error       174
symmetry error             172
fractal dimension error    163
worst radius               180
worst texture              168
worst perimeter            160
worst area                 173
worst smoothness           182
worst compactness          168
worst concavity            180
worst concave points       164
worst symmetry             173
worst fractal dimension    171
dtype: int64

(569, 30)

### MAR generation

Here we'll generate a dataset with MAR missingness across multiple columns simultaneously.

Missingness is introduced using the random method from mdatagen. In `n_xmiss=15` randomly selected column pairs, the rows with the lowest values of the observed column are made missing in the other column — so missingness depends on observed values, not on the missing value itself.

In [5]:
np.random.seed(42)
df_mar = mMAR(X=X, y=y, n_xmiss=15).random(missing_rate=30)

df_mar.isna().sum()

df_mar.shape

mean radius                  0
mean texture                 0
mean perimeter             341
mean area                    0
mean smoothness              0
mean compactness           341
mean concavity             341
mean concave points          0
mean symmetry                0
mean fractal dimension       0
radius error               341
texture error                0
perimeter error              0
area error                   0
smoothness error           341
compactness error          341
concavity error            341
concave points error       341
symmetry error             341
fractal dimension error    341
worst radius               341
worst texture                0
worst perimeter            341
worst area                   0
worst smoothness           341
worst compactness            0
worst concavity            341
worst concave points         0
worst symmetry               0
worst fractal dimension    341
target                       0
dtype: int64

(569, 31)

### MNAR generation

Here we'll generate a dataset with MNAR missingness across multiple columns simultaneously.

Missingness is generated on the prediction target `target` using `threshold=1`. Cases at the lower end of the target distribution are most likely to have their feature records missing, simulating the case where the most severe cases are systematically underrepresented in the data.

Since `target` is the target (`y`) and is absent from the generated feature matrix, its effect must be inferred through correlated proxies in the feature columns.

In [6]:
np.random.seed(42)
df_mnar = mMNAR(X=X, y=y, threshold=1, n_xmiss=15).random(missing_rate=30)

df_mnar.isna().sum()

df_mnar.shape

mean radius                341
mean texture                 0
mean perimeter             341
mean area                    0
mean smoothness              0
mean compactness           341
mean concavity             341
mean concave points        341
mean symmetry                0
mean fractal dimension       0
radius error                 0
texture error              341
perimeter error            341
area error                   0
smoothness error           341
compactness error            0
concavity error              0
concave points error       341
symmetry error               0
fractal dimension error      0
worst radius                 0
worst texture              341
worst perimeter              0
worst area                   0
worst smoothness           341
worst compactness          341
worst concavity              0
worst concave points       341
worst symmetry             341
worst fractal dimension    341
target                       0
dtype: int64

(569, 31)

Now we save the generated datasets to CSV so the analysis notebook can load them any time.

In [7]:
df_mcar.to_csv("mcar_multi.csv", index=False)
df_mar.to_csv("mar_multi.csv", index=False)
df_mnar.to_csv("mnar_multi.csv", index=False)